# B3 · Regresión lineal y logística desde cero

**Nivel Intermedio** · Trabajo práctico B3 · Punto de control del bloque · Se entrega al inicio de la clase de la semana 7 · Tiempo estimado: 4–5 horas · CPU (no hace falta GPU)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/melvinpqbsc/Alto_Rendimiento_IA/blob/main/Nivel_Intermedio/tareas/B3_regresion_desde_cero.ipynb)

Hasta ahora, `fit` ha sido una caja cerrada. En este trabajo la abres: programas regresión lineal, regresión logística y *softmax* con NumPy y descenso de gradiente, con la misma forma de usar que sklearn, y comparas tus resultados con los de sklearn **sobre la misma partición de los datos**. Si tu código está bien, los números deben coincidir. Cuando no coincidan, tendrás que averiguar por qué: en este trabajo hay una diferencia que no es un error tuyo.

Es el punto de control del bloque porque contiene todas las ideas del Bloque 2. Una red neuronal es esto mismo, con más capas: un modelo, una pérdida, un gradiente y un paso.

**Objetivos.** Al terminar podrás:

1. Derivar y programar sin bucles el gradiente de una pérdida, y verificarlo contra un gradiente numérico.
2. Implementar un modelo con `fit` y `predict`, y compararlo con sklearn y con una línea base usando la misma partición.
3. Explicar con tus datos qué pasa cuando la tasa de aprendizaje es demasiado grande o la pérdida parece haber terminado sin que los pesos hayan llegado.
4. Explicar qué hace la regularización y por qué sin ella los pesos pueden crecer sin límite.

**Prerrequisitos.** Clases en Python (semana 1), NumPy (semana 2), derivadas y descenso de gradiente (semana 4), y `fit/predict`, partición y métricas (semana 5). Si faltaste a alguna, la sección "Repaso rápido" de abajo trae lo necesario. No hace falta haber entregado ningún trabajo anterior.

**Cómo entregar.** Antes de escribir nada: *Archivo → Guardar una copia en GitHub*, repositorio `Alto-Rendimiento-IA/entregas-<tu-usuario>`, rama `main`, ruta **`B3/B3_regresion_desde_cero.ipynb`**, mensaje `Empezar B3`. Trabaja en la copia que se abre y guarda ahí después de cada ejercicio. Lo que esté en tu repositorio al inicio de la clase de la semana 7 es tu entrega.

**Uso de IA.** Puedes preguntarle a una IA qué significa un error o cómo se usa `np.linalg.eigvalsh`. No le pidas las derivadas, el código de los modelos ni las respuestas de las preguntas: ese es el ejercicio.

## Preparación

Ejecuta esta celda primero. Carga las librerías y los tres conjuntos de datos, y los separa en entrenamiento y prueba. **Esa partición queda fija**: tus modelos y los de sklearn verán exactamente las mismas filas. También define `numerical_grad`, que usarás para verificar tus gradientes.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer, load_diabetes, load_digits
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 0
rng = np.random.default_rng(SEED)

# Regresión: el avance de la diabetes un año después, a partir de 10 medidas de 442 pacientes
diabetes = load_diabetes(scaled=False)
X_dia_train, X_dia_test, y_dia_train, y_dia_test = train_test_split(
    diabetes.data, diabetes.target, test_size=0.25, random_state=SEED)

# Clasificación de dos clases: ¿el tumor es maligno (1) o benigno (0)? 30 medidas de 569 tumores
cancer = load_breast_cancer()
y_cancer = 1 - cancer.target  # en sklearn, 0 es maligno: lo invertimos para que 1 sea el caso que buscamos
X_can_train, X_can_test, y_can_train, y_can_test = train_test_split(
    cancer.data, y_cancer, test_size=0.25, stratify=y_cancer, random_state=SEED)

# Clasificación de diez clases: imágenes de 8 × 8 de dígitos escritos a mano
digits = load_digits()
X_dig_train, X_dig_test, y_dig_train, y_dig_test = train_test_split(
    digits.data, digits.target, test_size=0.25, stratify=digits.target, random_state=SEED)


def numerical_grad(f, p, h=1e-5):
    """Gradiente numérico de f en p, con diferencia centrada. p puede tener cualquier forma."""
    p = np.asarray(p, dtype=float)
    grad = np.zeros(p.shape)
    for i in np.ndindex(p.shape):
        up, down = p.copy(), p.copy()
        up[i] += h
        down[i] -= h
        grad[i] = (f(up) - f(down)) / (2 * h)
    return grad


print("diabetes:", X_dia_train.shape, X_dia_test.shape)
print("tumores: ", X_can_train.shape, X_can_test.shape)
print("dígitos: ", X_dig_train.shape, X_dig_test.shape)

## Repaso rápido

- **Regresión lineal.** Predice un número: $\hat{y} = Xw + b$, con $X$ de forma $(n, d)$, $w$ de forma $(d,)$ y $b$ un número. Se entrena minimizando el **error cuadrático medio** (MSE): $L = \frac{1}{n} \sum_i (\hat{y}_i - y_i)^2$.
- **Descenso de gradiente.** Desde un punto inicial, se repite $\theta \leftarrow \theta - \eta \, \nabla L(\theta)$. El número $\eta$ es la **tasa de aprendizaje** (*learning rate*).
- **Regresión logística.** Clasifica en dos clases. Calcula $z = Xw + b$ y lo convierte en una probabilidad con la **sigmoide**, $p = \sigma(z) = \dfrac{1}{1 + e^{-z}}$, que cumple $\sigma'(z) = \sigma(z)\,(1 - \sigma(z))$. Se predice 1 si $p \ge 0.5$. Se entrena minimizando la **entropía cruzada binaria** (*binary cross-entropy*): $L = -\frac{1}{n}\sum_i \big[ y_i \ln p_i + (1 - y_i) \ln(1 - p_i) \big]$.
- **Partición.** El modelo se ajusta con **entrenamiento** (*train*) y se mide con **prueba** (*test*), datos que nunca vio.
- **Estandarizar.** A cada columna se le resta su media y se la divide por su desviación estándar, ambas calculadas **solo con entrenamiento**. Así todas las columnas quedan en la misma escala y el descenso de gradiente funciona mucho mejor.
- **Línea base.** Lo mínimo a superar: predecir siempre la media (regresión) o la clase más común (clasificación).
- **Métricas.** Con VP verdaderos positivos, FP falsos positivos y FN falsos negativos: *accuracy* = fracción de aciertos, precisión = VP / (VP + FP), *recall* = VP / (VP + FN), y F1 es la media armónica de precisión y *recall*.
- **Una clase con `fit` y `predict`.** Así se ve el patrón de sklearn. `fit` guarda lo aprendido en atributos que terminan en `_` y devuelve `self`:

In [ ]:
class MeanPredictor:
    """Una línea base hecha a mano: siempre predice la media del entrenamiento."""

    def fit(self, X, y):
        self.mean_ = y.mean()
        return self

    def predict(self, X):
        return np.full(len(X), self.mean_)


print(MeanPredictor().fit(X_dia_train, y_dia_train).predict(X_dia_test[:3]))

## Ejercicio 1: estandarizar sin fuga de datos

**1a.** Completa la clase `Standardizer`:

- `fit(X)` guarda en `mean_` y `scale_` la media y la desviación estándar de cada columna (`axis=0`), y devuelve `self`. Si una columna no varía (desviación 0), su `scale_` debe valer 1, para no dividir por cero.
- `transform(X)` devuelve la $X$ estandarizada usando lo guardado.

Sin bucles.

In [ ]:
class Standardizer:
    def fit(self, X):
        # TODO 1a
        raise NotImplementedError

    def transform(self, X):
        # TODO 1a
        raise NotImplementedError

In [ ]:
for X_tr, X_te in [(X_dia_train, X_dia_test), (X_can_train, X_can_test), (X_dig_train, X_dig_test)]:
    mine = Standardizer().fit(X_tr)
    ref = StandardScaler().fit(X_tr)
    assert mine.mean_.shape == (X_tr.shape[1],) and mine.scale_.shape == (X_tr.shape[1],), "una media y una escala por columna"
    assert np.allclose(mine.transform(X_te), ref.transform(X_te)), "no coincide con StandardScaler"
assert (Standardizer().fit(X_dig_train).scale_ == 1).sum() == 4, "en los dígitos hay 4 columnas sin variación: su escala debe ser 1"
print("✓ Standardizer coincide con StandardScaler")

# Estos son los datos que usa el resto del trabajo
scaler_dia = Standardizer().fit(X_dia_train)
A_train, A_test = scaler_dia.transform(X_dia_train), scaler_dia.transform(X_dia_test)
scaler_can = Standardizer().fit(X_can_train)
B_train, B_test = scaler_can.transform(X_can_train), scaler_can.transform(X_can_test)
scaler_dig = Standardizer().fit(X_dig_train)
C_train, C_test = scaler_dig.transform(X_dig_train), scaler_dig.transform(X_dig_test)

**1b.** ¿Por qué `transform` usa la media y la escala del entrenamiento también para la prueba, en vez de calcular otras con los datos de prueba? ¿Qué columnas de los dígitos crees que no varían, y qué significan en una imagen?

*Tu respuesta:*

-

**Guarda en GitHub** con el mensaje `Ejercicio 1 resuelto`.

## Ejercicio 2: la pérdida y el gradiente de la regresión lineal

Para un solo paciente, con error $r_i = x_i \cdot w + b - y_i$, la pérdida es $r_i^2$. Por la regla de la cadena:

$$\frac{\partial (r_i^2)}{\partial w_j} = 2 r_i x_{ij} \qquad\qquad \frac{\partial (r_i^2)}{\partial b} = 2 r_i$$

La pérdida total promedia sobre los $n$ pacientes. **Tu trabajo es escribir esos promedios sin bucles**, de una vez para todas las filas y columnas. Piensa en las formas: `r` es $(n,)$, `X` es $(n, d)$ y el resultado para $w$ debe ser $(d,)$.

**2a.** Completa las tres funciones. `mse_grad` devuelve dos cosas: el gradiente respecto de $w$, de forma $(d,)$, y el de $b$, un número.

In [ ]:
def linear_predict(X, w, b):
    # TODO 2a
    raise NotImplementedError


def mse(y, pred):
    # TODO 2a
    raise NotImplementedError


def mse_grad(X, y, w, b):
    # TODO 2a
    raise NotImplementedError

In [ ]:
w_try, b_try = rng.normal(size=A_train.shape[1]), 3.0
pred = linear_predict(A_train, w_try, b_try)
assert pred.shape == (A_train.shape[0],), f"linear_predict debería dar forma {(A_train.shape[0],)}, da {pred.shape}"
assert np.isclose(mse(np.array([1.0, 2.0]), np.array([2.0, 4.0])), 2.5), "el MSE de ([1, 2], [2, 4]) es (1 + 4) / 2 = 2.5"

dw, db = mse_grad(A_train, y_dia_train, w_try, b_try)
assert np.shape(dw) == (A_train.shape[1],), f"el gradiente de w debería tener forma {(A_train.shape[1],)}, tiene {np.shape(dw)}"
assert np.ndim(db) == 0, "el gradiente de b debe ser un solo número"
num_w = numerical_grad(lambda v: mse(y_dia_train, linear_predict(A_train, v, b_try)), w_try)
num_b = numerical_grad(lambda v: mse(y_dia_train, linear_predict(A_train, w_try, v[0])), [b_try])[0]
assert np.allclose(dw, num_w, atol=1e-3), "el gradiente de w no coincide con el numérico"
assert np.isclose(db, num_b, atol=1e-3), "el gradiente de b no coincide con el numérico"
print("✓ el gradiente coincide con el numérico")

**Guarda en GitHub** con el mensaje `Ejercicio 2 resuelto`.

## Ejercicio 3: entrenar, y encontrar el límite de la tasa de aprendizaje

**3a.** Completa `MyLinearRegression`, con la misma forma de usar que `LinearRegression` de sklearn:

- `fit(X, y)` empieza con $w = 0$, $b = 0$ y da `steps` pasos de descenso de gradiente. Guarda la pérdida de **antes** de cada paso en la lista `losses_`, y lo aprendido en `coef_` e `intercept_`. Devuelve `self`.
- `predict(X)` usa `coef_` e `intercept_`.

In [ ]:
class MyLinearRegression:
    def __init__(self, lr=0.1, steps=5000):
        self.lr = lr
        self.steps = steps

    def fit(self, X, y):
        # TODO 3a
        raise NotImplementedError

    def predict(self, X):
        # TODO 3a
        raise NotImplementedError

In [ ]:
X_toy = np.array([[0.0], [1.0], [2.0], [3.0]])
toy = MyLinearRegression(lr=0.1, steps=2000).fit(X_toy, 2 * X_toy[:, 0] + 1)
assert np.isclose(toy.coef_[0], 2, atol=1e-3) and np.isclose(toy.intercept_, 1, atol=1e-3), "debería aprender y = 2x + 1"
assert len(toy.losses_) == 2000, "losses_ debe tener una pérdida por paso"
assert toy.losses_[0] > toy.losses_[-1], "la pérdida debería bajar"
assert np.allclose(toy.predict(np.array([[10.0]])), 21, atol=1e-2), "predict(10) debería dar 21"
print("✓ MyLinearRegression aprende y = 2x + 1")

**3b.** Con `A_train` y `y_dia_train`, entrena 300 pasos con cada tasa de `lrs` y dibuja las cinco curvas de pérdida en **un** gráfico, con eje $y$ logarítmico, ejes con nombre y leyenda. Una tasa que diverge da números enormes y avisos de desbordamiento: envuelve el entrenamiento en `with np.errstate(all="ignore"):`. Usa `ax.set_ylim(2000, 1e6)` para que se vean todas.

In [ ]:
lrs = [0.001, 0.01, 0.1, 0.2, 0.25]
# TODO 3b

**3c.** ¿Dónde está exactamente el límite? En una parábola $\frac{c}{2}x^2$, el descenso de gradiente converge si $\eta < 2/c$, donde $c$ es la curvatura. El MSE es una parábola en muchas dimensiones, y su **curvatura máxima** es el valor propio más grande de la matriz $\frac{2}{n} X^\top X$. Calcula ese valor con `np.linalg.eigvalsh(...)`, que devuelve todos los valores propios de una matriz simétrica, y guarda en `lr_limit` la tasa máxima que predice la teoría.

**3d.** Compruébalo: guarda en `loss_below` la pérdida final de 2000 pasos con una tasa un poco **menor** que el límite, 0.24, y en `loss_above` la de una un poco **mayor**, 0.25.

In [ ]:
lr_limit = None    # TODO 3c
loss_below = None  # TODO 3d
loss_above = None  # TODO 3d
print(lr_limit, loss_below, loss_above)

In [ ]:
assert lr_limit is not None and loss_below is not None and loss_above is not None, "completa 3c y 3d"
assert 0.24 < lr_limit < 0.25, f"lr_limit = {lr_limit}: revisa el valor propio más grande de 2/n · XᵀX"
assert loss_below < 3000, "con 0.24 debería converger"
assert not np.isfinite(loss_above) or loss_above > 1e6, "con 0.25 debería diverger"
print(f"✓ el límite es {lr_limit:.4f}: justo debajo converge y justo encima diverge")

**Guarda en GitHub** con el mensaje `Ejercicio 3 resuelto`.

## Ejercicio 4: contra sklearn, sobre la misma partición

**4a.** Los tres modelos de `reg_models` se usan igual: esa es la ventaja de tener la misma forma. Recórrelos con un solo bucle: entrena cada uno con `A_train, y_dia_train` y arma `reg_table`, un `DataFrame` con una fila por modelo (el índice es el nombre) y las columnas `mse_train`, `mse_test` y `r2_test`. El $R^2$ es la fracción de la varianza que el modelo explica: $R^2 = 1 - \text{MSE} / \text{Var}(y)$, con los datos de prueba.

In [ ]:
reg_models = {
    "línea base (la media)": DummyRegressor(strategy="mean"),
    "mi descenso de gradiente": MyLinearRegression(lr=0.1, steps=5000),
    "sklearn": LinearRegression(),
}
reg_table = None  # TODO 4a
reg_table

In [ ]:
assert reg_table is not None, "completa 4a"
assert list(reg_table.columns) == ["mse_train", "mse_test", "r2_test"], "columnas: mse_train, mse_test, r2_test"
sk_ref = LinearRegression().fit(A_train, y_dia_train)
assert np.isclose(reg_table.loc["sklearn", "mse_test"], mse(y_dia_test, sk_ref.predict(A_test))), "revisa mse_test de sklearn"
assert abs(reg_table.loc["mi descenso de gradiente", "mse_test"] - reg_table.loc["sklearn", "mse_test"]) < 0.1, "tu MSE debería coincidir con el de sklearn"
gd_model = reg_models["mi descenso de gradiente"]
assert np.allclose(gd_model.coef_, sk_ref.coef_, atol=0.01), "tus pesos deberían coincidir con los de sklearn"
assert np.isclose(gd_model.intercept_, sk_ref.intercept_, atol=0.01), "tu sesgo debería coincidir con el de sklearn"
assert reg_table.loc["sklearn", "mse_test"] < reg_table.loc["línea base (la media)", "mse_test"], "debería superar a la línea base"
print("✓ tu regresión lineal coincide con la de sklearn y supera a la línea base")

**4b.** Una trampa. Entrena `slow = MyLinearRegression(lr=0.01, steps=5000)`. Guarda en `loss_gap` cuánto más alta queda su pérdida de entrenamiento que la de sklearn, como fracción (0.01 sería 1%), y en `coef_gap` la mayor diferencia entre sus pesos y los de sklearn (en valor absoluto). Mira también qué tan correlacionadas están las columnas `s1` y `s2` con `np.corrcoef`, usando los nombres de `diabetes.feature_names`.

In [ ]:
slow = None      # TODO 4b
loss_gap = None  # TODO 4b
coef_gap = None  # TODO 4b
print(loss_gap, coef_gap)

In [ ]:
assert slow is not None and loss_gap is not None and coef_gap is not None, "completa 4b"
assert 0 < loss_gap < 0.002, f"loss_gap = {loss_gap}: debería ser menor que 0.2%"
assert coef_gap > 5, f"coef_gap = {coef_gap}: debería ser mayor que 5"
print(f"✓ la pérdida queda a {loss_gap:.2%}, pero un peso se equivoca por {coef_gap:.1f}")

**4c.** Responde con tus números:

1. ¿Cuánto mejora el modelo a la línea base en la prueba? ¿Te parece mucho, con ese $R^2$?
2. En 4b, la pérdida casi terminó y los pesos no. ¿Cómo puede ser? ¿Qué tiene que ver con que `s1` y `s2` estén tan correlacionadas? (Piensa en la forma del valle de la pérdida: empinado en unas direcciones y casi plano en otras.)
3. ¿Qué lección sacas para interpretar los pesos de un modelo, por ejemplo para decir "esta medida es la que más importa"?

*Tus respuestas:*

- 1.
- 2.
- 3.

**Guarda en GitHub** con el mensaje `Ejercicio 4 resuelto`.

## Ejercicio 5: la pérdida y el gradiente de la regresión logística

Ahora los tumores: $y = 1$ es maligno. Para un solo tumor, con $z = x \cdot w + b$ y $p = \sigma(z)$, la pérdida es $\ell = -y \ln p - (1 - y) \ln(1 - p)$.

**5a. En papel.** Usa la regla de la cadena y $\sigma'(z) = p(1 - p)$ para demostrar que $\dfrac{\partial \ell}{\partial z} = p - y$. Después, como $\dfrac{\partial z}{\partial w_j} = x_j$ y $\dfrac{\partial z}{\partial b} = 1$, escribe $\dfrac{\partial \ell}{\partial w_j}$ y $\dfrac{\partial \ell}{\partial b}$.

*Tu derivación (puedes escribirla en LaTeX entre signos `$`, o subir una foto del papel):*

-

**5b.** Completa las funciones, sin bucles. `sigmoid` ya está. `binary_cross_entropy(y, p)` debe recortar `p` con `np.clip(p, 1e-12, 1 - 1e-12)` antes del logaritmo, porque $\ln 0$ no existe. `logistic_grad(X, y, w, b)` promedia tu derivada de 5a sobre los $n$ tumores y devuelve el gradiente de $w$ (forma $(d,)$) y el de $b$ (un número).

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))  # el recorte evita desbordar la exponencial


def binary_cross_entropy(y, p):
    # TODO 5b
    raise NotImplementedError


def logistic_grad(X, y, w, b):
    # TODO 5b
    raise NotImplementedError

In [ ]:
assert np.isclose(binary_cross_entropy(np.array([1, 0]), np.array([0.5, 0.5])), np.log(2)), "con p = 0.5 la pérdida es ln 2"
assert binary_cross_entropy(np.array([1.0]), np.array([0.0])) < 30, "falta recortar p: ln 0 no existe"
w_try, b_try = rng.normal(size=B_train.shape[1]) / 3, 0.5
dw, db = logistic_grad(B_train, y_can_train, w_try, b_try)
assert np.shape(dw) == (B_train.shape[1],), f"el gradiente de w debería tener forma {(B_train.shape[1],)}, tiene {np.shape(dw)}"
assert np.ndim(db) == 0, "el gradiente de b debe ser un solo número"
num_w = numerical_grad(lambda v: binary_cross_entropy(y_can_train, sigmoid(B_train @ v + b_try)), w_try)
num_b = numerical_grad(lambda v: binary_cross_entropy(y_can_train, sigmoid(B_train @ w_try + v[0])), [b_try])[0]
assert np.allclose(dw, num_w, atol=1e-5), "el gradiente de w no coincide con el numérico"
assert np.isclose(db, num_b, atol=1e-5), "el gradiente de b no coincide con el numérico"
print("✓ el gradiente coincide con el numérico")

**5c. ¿Por qué no usar MSE?** Un tumor maligno ($y = 1$) y un modelo muy seguro de lo contrario: $z = -6$. Compara cuánto empuja cada pérdida. Con entropía cruzada, $\dfrac{\partial \ell}{\partial z} = p - y$. Con MSE, $(p - y)^2$, la regla de la cadena da $2(p - y)\,p(1 - p)$.

Completa `grad_ce(z, y)` y `grad_mse(z, y)`, y dibuja el **valor absoluto** de las dos en un gráfico, para $z$ entre −8 y 8 con $y = 1$, con ejes y leyenda.

In [ ]:
def grad_ce(z, y):
    # TODO 5c
    raise NotImplementedError


def grad_mse(z, y):
    # TODO 5c
    raise NotImplementedError


# TODO 5c: el gráfico

In [ ]:
z_check = np.array([-6.0, 0.0, 2.0])
num_ce = [numerical_grad(lambda v: binary_cross_entropy(np.array([1.0]), sigmoid(v)), [z])[0] for z in z_check]
num_mse = [numerical_grad(lambda v: (sigmoid(v[0]) - 1) ** 2, [z])[0] for z in z_check]
assert np.allclose(grad_ce(z_check, 1), num_ce, atol=1e-5), "grad_ce no coincide con la derivada numérica"
assert np.allclose(grad_mse(z_check, 1), num_mse, atol=1e-5), "grad_mse no coincide con la derivada numérica"
print("✓ las dos derivadas coinciden con las numéricas")

**5d.** En $z = -6$, ¿cuántas veces más empuja la entropía cruzada que MSE? ¿Por qué eso es lo contrario de lo que querríamos de MSE, justo cuando el modelo está más equivocado?

*Tu respuesta:*

-

**Guarda en GitHub** con el mensaje `Ejercicio 5 resuelto`.

## Ejercicio 6: la diferencia con sklearn

**6a.** Completa `MyLogisticRegression`, con la misma forma de usar que sklearn:

- `fit(X, y)` empieza con $w = 0$, $b = 0$, da `steps` pasos y guarda `losses_`, `coef_` e `intercept_`. Por ahora **ignora** `lam`.
- `predict_proba(X)` devuelve un arreglo de forma $(n, 2)$, como en sklearn: la columna 0 es la probabilidad de la clase 0, y la 1 la de la clase 1.
- `predict(X)` devuelve 1 donde la probabilidad de la clase 1 es al menos 0.5, y 0 en el resto, como enteros.

In [ ]:
class MyLogisticRegression:
    def __init__(self, lr=0.5, steps=5000, lam=0.0):
        self.lr = lr
        self.steps = steps
        self.lam = lam

    def fit(self, X, y):
        # TODO 6a (y 6c)
        raise NotImplementedError

    def predict_proba(self, X):
        # TODO 6a
        raise NotImplementedError

    def predict(self, X):
        # TODO 6a
        raise NotImplementedError

In [ ]:
plain = MyLogisticRegression(lr=0.5, steps=5000).fit(B_train, y_can_train)
proba = plain.predict_proba(B_test)
assert proba.shape == (len(B_test), 2), f"predict_proba debería tener forma {(len(B_test), 2)}, tiene {proba.shape}"
assert np.allclose(proba.sum(axis=1), 1), "cada fila de predict_proba debe sumar 1"
assert set(np.unique(plain.predict(B_test))) <= {0, 1}, "predict debe devolver 0 o 1"
assert plain.losses_[-1] < 0.06 and len(plain.losses_) == 5000, "la pérdida final debería ser menor que 0.06, con una pérdida por paso"
print(f"✓ MyLogisticRegression entrena: la pérdida baja de {plain.losses_[0]:.3f} a {plain.losses_[-1]:.3f}")

**6b.** Completa `metrics_row(model, X_tr, y_tr, X_te, y_te)`: entrena el modelo y devuelve un diccionario con `acc_train`, `acc_test`, `precision`, `recall` y `f1` (estas tres, en la prueba). Con esa función, arma `clf_table` para los tres modelos de `clf_models`. En `precision_score`, usa `zero_division=0`: la línea base nunca dice "maligno".

In [ ]:
def metrics_row(model, X_tr, y_tr, X_te, y_te):
    # TODO 6b
    raise NotImplementedError


clf_models = {
    "línea base (la clase más común)": DummyClassifier(strategy="most_frequent"),
    "mi descenso de gradiente": MyLogisticRegression(lr=0.5, steps=5000),
    "sklearn": LogisticRegression(),
}
clf_table = pd.DataFrame({name: metrics_row(model, B_train, y_can_train, B_test, y_can_test)
                          for name, model in clf_models.items()}).T
clf_table.round(3)

In [ ]:
assert list(clf_table.columns) == ["acc_train", "acc_test", "precision", "recall", "f1"], "revisa las columnas de metrics_row"
assert clf_table.loc["línea base (la clase más común)", "f1"] == 0, "la línea base nunca dice maligno: su F1 es 0"
assert clf_table.loc["mi descenso de gradiente", "acc_test"] > 0.95, "tu modelo debería superar el 95% en la prueba"
w_sk = clf_models["sklearn"].coef_[0]
print("norma de tus pesos:     ", np.linalg.norm(clf_models["mi descenso de gradiente"].coef_).round(2))
print("norma de los de sklearn:", np.linalg.norm(w_sk).round(2))
print("mayor diferencia entre pesos:", np.abs(clf_models["mi descenso de gradiente"].coef_ - w_sk).max().round(2))

**No coinciden**, y tus gradientes estaban bien. Antes de seguir, revisa la documentación de [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) y mira los parámetros `penalty` y `C`.

**6c. La regularización.** sklearn no minimiza solo la entropía cruzada. Por defecto agrega una penalización a los pesos grandes (*L2*). Su objetivo es $C \sum_i \ell_i + \frac{1}{2} \|w\|^2$. Si lo divides por $Cn$ (dividir por una constante no mueve el mínimo), queda:

$$L = \frac{1}{n} \sum_i \ell_i + \frac{\lambda}{2} \|w\|^2 \qquad \text{con} \quad \lambda = \frac{1}{Cn}$$

El sesgo $b$ no se penaliza. Agrega `lam` a tu `fit`: suma el término a `losses_` y su gradiente, $\lambda w$, al gradiente de $w$. Vuelve a ejecutar las celdas de 6a. Después, crea `reg` con el `lam` que corresponde a $C = 1$, la opción por defecto de sklearn, y agrégalo a `clf_table` con el nombre `"mi descenso de gradiente, con lam"`.

In [ ]:
lam_sk = None  # TODO 6c: λ para C = 1 y estos datos
reg = None     # TODO 6c: MyLogisticRegression(lr=0.5, steps=5000, lam=lam_sk), entrenado
# TODO 6c: agrega la fila a clf_table
clf_table.round(3)

In [ ]:
assert lam_sk is not None and reg is not None, "completa 6c"
assert np.isclose(lam_sk, 1 / len(B_train)), "con C = 1, λ = 1 / n"
assert np.allclose(reg.coef_, w_sk, atol=0.05), f"los pesos deberían coincidir con sklearn; la mayor diferencia es {np.abs(reg.coef_ - w_sk).max():.3f}"
assert np.isclose(reg.intercept_, clf_models["sklearn"].intercept_[0], atol=0.05), "el sesgo debería coincidir con sklearn"
assert (reg.predict(B_test) == clf_models["sklearn"].predict(B_test)).all(), "las predicciones deberían ser las mismas"
assert "mi descenso de gradiente, con lam" in clf_table.index, "falta la fila en clf_table"
print("✓ con la penalización, tu modelo y el de sklearn coinciden")

**6d. ¿Qué hace la penalización?** Para `steps` en `[1000, 5000, 20000]`, entrena dos modelos, uno sin penalización y otro con `lam_sk`, con `lr=0.5`. Arma `growth`, un `DataFrame` con una fila por modelo (las columnas `steps`, `lam`, `norm`, `loss_ce`, `acc_train` y `acc_test`), donde `norm` es `np.linalg.norm(coef_)` y `loss_ce` la entropía cruzada de entrenamiento **sin** penalización.

Agrega una fila más con `LogisticRegression(penalty=None, max_iter=5000)` de sklearn, para ver adónde va el modelo sin penalización si se deja que termine (usa `steps` igual a su `n_iter_[0]`).

In [ ]:
growth = None  # TODO 6d
growth

In [ ]:
assert growth is not None and len(growth) == 7, "growth: 6 modelos tuyos y 1 de sklearn"
assert {"steps", "lam", "norm", "loss_ce", "acc_train", "acc_test"} <= set(growth.columns), "revisa las columnas"
mine_steps = growth["steps"].isin([1000, 5000, 20000])
no_pen = growth[mine_steps & (growth["lam"] == 0)].sort_values("steps")
pen = growth[mine_steps & (growth["lam"] > 0)].sort_values("steps")
assert len(no_pen) == 3 and len(pen) == 3, "tres modelos sin penalización y tres con ella, con steps 1000, 5000 y 20000"
assert no_pen["norm"].is_monotonic_increasing, "sin penalización, la norma debería crecer con los pasos"
assert growth["norm"].max() > 100, "falta el modelo de sklearn sin penalización, con pesos enormes"
assert pen["norm"].max() - pen["norm"].min() < 0.5, "con penalización, la norma debería quedarse casi quieta"
print("✓ sin penalización los pesos siguen creciendo; con penalización se detienen")

**6e.** Responde con tus números:

1. ¿Por qué tu primer modelo no coincidía con sklearn? ¿Qué aprendes sobre comparar tu código con una librería?
2. El modelo de sklearn sin penalización acierta el 100% del entrenamiento. Sin penalización, ¿por qué los pesos crecen sin parar y la pérdida sigue bajando? (Pista: si una recta o un plano separa todos los puntos, ¿qué pasa con la pérdida si multiplicas $w$ y $b$ por 2?)
3. ¿Con cuál modelo te quedarías para el hospital? ¿Qué relación tiene esto con el sobreajuste de la semana 5?

*Tus respuestas:*

- 1.
- 2.
- 3.

**Guarda en GitHub** con el mensaje `Ejercicio 6 resuelto`.

## Ejercicio 7: varias clases con *softmax*

Con diez dígitos, cada clase tiene su propio puntaje. El modelo es una capa lineal seguida de *softmax*: $Z = XW + b$, con $W$ de forma $(d, k)$ y $b$ de forma $(k,)$, y $P = \text{softmax}(Z)$. La pérdida es la entropía cruzada, $L = -\frac{1}{n}\sum_i \ln P_{i, y_i}$.

Con $Y$ las respuestas en *one-hot* (una fila por imagen, con un 1 en la columna de su clase) y $G = (P - Y)/n$, los gradientes son:

$$\nabla_W L = X^\top G \qquad\qquad \nabla_b L = \sum_i G_i$$

Es la misma cuenta $p - y$ de la regresión logística, una columna por clase.

**7a.** Completa `softmax_grad(X, Y, W, b)` y `MySoftmaxRegression`, con la misma forma que `MyLogisticRegression`: `fit(X, y)` recibe las clases como enteros y las convierte a *one-hot* con `np.eye(k)[y]`, y usa `lam` como en 6c. `predict` devuelve la clase con mayor puntaje. No hace falta `predict_proba`.

In [ ]:
def softmax(Z):
    E = np.exp(Z - Z.max(axis=1, keepdims=True))
    return E / E.sum(axis=1, keepdims=True)


def softmax_grad(X, Y, W, b):
    # TODO 7a
    raise NotImplementedError


class MySoftmaxRegression:
    def __init__(self, lr=1.0, steps=2000, lam=0.0):
        self.lr = lr
        self.steps = steps
        self.lam = lam

    def fit(self, X, y):
        # TODO 7a
        raise NotImplementedError

    def predict(self, X):
        # TODO 7a
        raise NotImplementedError

In [ ]:
X_small, y_small = C_train[:40], y_dig_train[:40]
Y_small = np.eye(10)[y_small]
W_try, b_try = rng.normal(size=(64, 10)) / 5, rng.normal(size=10) / 5
dW, db = softmax_grad(X_small, Y_small, W_try, b_try)
assert np.shape(dW) == (64, 10) and np.shape(db) == (10,), f"formas {np.shape(dW)} y {np.shape(db)}: esperaba (64, 10) y (10,)"


def softmax_loss(W, b):
    return -np.log(softmax(X_small @ W + b)[np.arange(40), y_small]).mean()


assert np.allclose(dW, numerical_grad(lambda v: softmax_loss(v, b_try), W_try), atol=1e-5), "el gradiente de W no coincide con el numérico"
assert np.allclose(db, numerical_grad(lambda v: softmax_loss(W_try, v), b_try), atol=1e-5), "el gradiente de b no coincide con el numérico"
print("✓ el gradiente coincide con el numérico")

**7b.** Entrena `soft` con los dígitos ($C = 1$, `lr=1.0`, `steps=2000`, y `lam` calculada para estos datos). Con `metrics_row` no se puede (la precisión de sklearn pide otras opciones con varias clases), así que guarda en `digit_scores`, un diccionario, el *accuracy* de prueba de la línea base `DummyClassifier(strategy="most_frequent")`, de `soft` y de `LogisticRegression(max_iter=1000)` de sklearn, con las claves `"base"`, `"mine"` y `"sklearn"`.

In [ ]:
soft = None          # TODO 7b
sk_digits = None     # TODO 7b
digit_scores = None  # TODO 7b
print(digit_scores)

In [ ]:
assert digit_scores is not None and soft is not None and sk_digits is not None, "completa 7b"
assert digit_scores["base"] < 0.15, "la línea base debería acertar cerca del 10%"
assert abs(digit_scores["mine"] - digit_scores["sklearn"]) < 0.01, "tu accuracy debería coincidir con el de sklearn"
agreement = np.mean(soft.predict(C_test) == sk_digits.predict(C_test))
assert agreement > 0.98, f"tus predicciones coinciden con sklearn solo en {agreement:.1%}"
assert soft.coef_.shape == (64, 10), "W debería tener forma (64, 10)"
print(f"✓ tu softmax y el de sklearn coinciden en {agreement:.1%} de las predicciones")

**7c.** Cada columna de $W$ tiene 64 números, uno por píxel: se puede dibujar como una imagen de 8 × 8. Dibuja las diez en una grilla de 2 × 5 con `imshow(..., cmap="RdBu_r", vmin=-m, vmax=m)`, donde `m` es el mayor valor absoluto de `soft.coef_`: así el 0 queda en blanco. Cada imagen lleva como título su dígito, sin ejes (`ax.axis("off")`). Rojo significa que ese píxel encendido suma votos para el dígito; azul, que resta.

In [ ]:
# TODO 7c

**Guarda en GitHub** con el mensaje `Ejercicio 7 resuelto`.

## Ejercicio 8: interpretar

Responde con tus números y tus gráficos, en dos a cuatro oraciones cada una.

1. **Tumores.** En tu `clf_table`, ¿cuál es el *recall* del mejor modelo, y cuántos tumores malignos de la prueba se le escapan? Para este problema, ¿qué error es más grave, un falso positivo o un falso negativo? ¿Qué cambiarías en `predict` si quisieras que se escaparan menos?
2. **Los pesos como imágenes.** Elige uno de tus dibujos de 7c. ¿Se parece al dígito? ¿Qué zonas ayudan y cuáles restan? ¿Qué limitación ves en un modelo que solo puede sumar píxeles con pesos?
3. **La receta.** Tus tres modelos tienen casi el mismo `fit`. Escribe en una lista qué cuatro cosas cambian de un modelo a otro y cuáles quedan iguales. Si quisieras agregar una capa oculta, ¿qué parte sería la difícil, y por qué? (Es el tema de la semana 7.)
4. **Tú contra sklearn.** Tu `fit` de 5000 pasos y el de sklearn dan lo mismo. ¿Por qué, entonces, se usa sklearn en la práctica? Mide el tiempo de los dos con `%time` en una de las regresiones si quieres apoyar tu respuesta con un número.

*Tus respuestas:*

- 1.
- 2.
- 3.
- 4.

**Guarda en GitHub** con el mensaje `Ejercicio 8 resuelto`.

## Conclusión

*Escribe aquí un párrafo en lenguaje sencillo: qué aprendiste al abrir la caja de `fit`, qué significan tus números comparados con la línea base y con sklearn, y qué te gustaría probar después. Sin este párrafo el trabajo no tiene nota.*

## Uso de IA

*Una línea: para qué usaste IA en este trabajo, o "No usé IA".*

## Rúbrica

Primero asígnate tu puntaje (columna "Yo"). Tu compañero revisor completa la columna "Revisor" con un comentario en la copia de Drive que le compartas.

| Criterio | Qué se mira | Yo (0–2) | Revisor (0–2) |
|---|---|---|---|
| **Se ejecuta** | *Entorno de ejecución → Reiniciar y ejecutar todo* corre sin errores | | |
| **Es correcto** | Todas las verificaciones imprimen ✓; los gradientes y los modelos no recorren filas con bucles (el único bucle es el de los pasos); los gráficos de 3b, 5c y 7c tienen lo pedido | | |
| **Está justificado** | La derivación de 5a y las respuestas de 1b, 4c, 5d, 6e y del ejercicio 8 explican el porqué con tus números y gráficos | | |
| **Está bien comunicado** | La conclusión se entiende sin leer el código; hay un *commit* por ejercicio con un mensaje claro | | |
| **Total** | | /8 | /8 |

**Antes de la clase de la semana 7**: ejecuta *Entorno de ejecución → Reiniciar y ejecutar todo*, revisa que todo imprima ✓ y guarda por última vez en `B3/B3_regresion_desde_cero.ipynb`.